In [1]:
!pip3 install googletrans==3.1.0a0

     |████████████████████████████████| 55 kB 1.3 MB/s 
     |████████████████████████████████| 1.3 MB 15.7 MB/s 
     |████████████████████████████████| 42 kB 738 kB/s 
     |████████████████████████████████| 65 kB 2.1 MB/s 
     |████████████████████████████████| 53 kB 1.5 MB/s 
  Created wheel for googletrans: filename=googletrans-3.1.0a0-py3-none-any.whl size=16367 sha256=32ee084630d8366fca8ba52f166d332b608a19d00223e504442ec5676d30525f
  Stored in directory: /root/.cache/pip/wheels/0c/be/fe/93a6a40ffe386e16089e44dad9018ebab9dc4cb9eb7eab65ae
Successfully built googletrans


In [2]:
pip install google_trans_new

In [3]:
pip install googletrans


In [4]:
pip install Sastrawi

     |████████████████████████████████| 209 kB 7.9 MB/s 


In [5]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [6]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

**IMPORT LIBRARY YANG DIBUTUHKAN**

In [7]:
import tweepy
from textblob import TextBlob
from wordcloud import WordCloud
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
from google.colab import drive
drive.mount('drive')

Mounted at drive


**READ DATASET**

In [8]:
data=pd.read_csv('/content/medictreat3.csv')
data

,date,username,fulltext
0,2022-03-08 15:53:56,dimplejaeeee,@bucinbabang @bertanyarl sebelumnya aku konsul...
1,2022-03-08 15:47:59,rezkyadit28_,@bika_engine @adams69apple @Blackgownn @tangan...
2,2022-03-08 15:47:10,salmooun,"udah lama ga ke limi, dateng2 susternya drku g..."
3,2022-03-08 15:39:22,jimbon90,@Baebbyz @isyour__crush @kochengfess Aku baru ...
4,2022-03-08 15:38:06,jaetimein,capek banget bisa bisanya meluk dokternya😭\r\n...
...,...,...,...
1740,2022-03-04 06:03:28,OktoFerdi,RT @tutudetoxic: @Akashi2105 @lazy_homer @iams...
1741,2022-03-04 06:00:19,Rieska_Ayu11,@SecuLitty @AngelenosLos @ddubokki @MitaCestal...
1742,2022-03-04 05:59:56,boredafhere,#Halodoc scam banget udh bayar pake debit aja ...
1743,2022-03-04 05:55:54,meithestars,"kmrn janjian dokter langsung buat hari selasa,..."


**DROP COLUMN WE DON'T NEEDED**

In [9]:
data.drop(['date','username'], axis=1, inplace=True)
data.rename(columns={'fulltext':'Tweets'})

,Tweets
0,@bucinbabang @bertanyarl sebelumnya aku konsul...
1,@bika_engine @adams69apple @Blackgownn @tangan...
2,"udah lama ga ke limi, dateng2 susternya drku g..."
3,@Baebbyz @isyour__crush @kochengfess Aku baru ...
4,capek banget bisa bisanya meluk dokternya😭\r\n...
...,...
1740,RT @tutudetoxic: @Akashi2105 @lazy_homer @iams...
1741,@SecuLitty @AngelenosLos @ddubokki @MitaCestal...
1742,#Halodoc scam banget udh bayar pake debit aja ...
1743,"kmrn janjian dokter langsung buat hari selasa,..."


**TAHAP PREPROCESSING**

In [10]:
import pandas as pd
import numpy as np
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import re
import unicodedata

In [11]:
# remove user from the text
def remove_user(input_txt, pattern):
    r = re.findall(pattern, input_txt)
    for i in r:
        input_txt = re.sub(i, '', input_txt)
    return input_txt    
data['tweet_remove_user'] = np.vectorize(remove_user)(data['fulltext'], "@[\w]*")
data

,fulltext,tweet_remove_user
0,@bucinbabang @bertanyarl sebelumnya aku konsul...,sebelumnya aku konsultasi dulu ke dokter yg ...
1,@bika_engine @adams69apple @Blackgownn @tangan...,Serius ms glow? Gua kira produk skincare y...
2,"udah lama ga ke limi, dateng2 susternya drku g...","udah lama ga ke limi, dateng2 susternya drku g..."
3,@Baebbyz @isyour__crush @kochengfess Aku baru ...,"Aku baru adopt kucing kak, eh tbtb dia diar..."
4,capek banget bisa bisanya meluk dokternya😭\r\n...,capek banget bisa bisanya meluk dokternya😭\r\n...
...,...,...
1740,RT @tutudetoxic: @Akashi2105 @lazy_homer @iams...,"RT : Tenang, perang di ukraina gak ngince..."
1741,@SecuLitty @AngelenosLos @ddubokki @MitaCestal...,Mama jawab ke dokternya klo mama dah ster...
1742,#Halodoc scam banget udh bayar pake debit aja ...,#Halodoc scam banget udh bayar pake debit aja ...
1743,"kmrn janjian dokter langsung buat hari selasa,...","kmrn janjian dokter langsung buat hari selasa,..."


In [12]:
# cleaning the emoji,retweet,etc
def cleaning(strTweet):
    #remove non-ascii
    strTweet = unicodedata.normalize('NFKD', strTweet).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    
    #remove URLs
    strTweet = re.sub(r'(?i)\b((?:https?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:\'".,<>?«»“”‘’]))', '', strTweet)
    
    #remove punctuations
    strTweet = re.sub(r'[^\w]|_',' ',strTweet)
    
    #remove digit from string
    strTweet = re.sub("\S*\d\S*", "", strTweet).strip()
    
    #remove digit or numbers
    strTweet = re.sub(r"\b\d+\b", " ", strTweet)
    
    #Remove additional white spaces
    strTweet = re.sub('[\s]+', ' ', strTweet)
    
    #remove rt
    strTweet = re.sub(r'^RT[\s]+', '', strTweet)
    
    #remove tab, new line, and backslice
    strTweet = strTweet.replace('\\t', " ").replace('\\n', " ").replace('\\u', " ").replace('\\', "")
    
    # remove whitespace leading & trailing
    strTweet = strTweet.strip()
    
    # remove multiple whitespace into single whitespace
    strTweet = re.sub('\s+',' ',strTweet)
    
    # remove single character
    strTweet = re.sub(r"\b[a-zA-Z]\b", "", strTweet)
    
    return strTweet
data['tweet_cleaning'] = data['tweet_remove_user'].apply(cleaning)
data

,fulltext,tweet_remove_user,tweet_cleaning
0,@bucinbabang @bertanyarl sebelumnya aku konsul...,sebelumnya aku konsultasi dulu ke dokter yg ...,sebelumnya aku konsultasi dulu ke dokter yg du...
1,@bika_engine @adams69apple @Blackgownn @tangan...,Serius ms glow? Gua kira produk skincare y...,Serius ms glow Gua kira produk skincare yg kas...
2,"udah lama ga ke limi, dateng2 susternya drku g...","udah lama ga ke limi, dateng2 susternya drku g...",udah lama ga ke limi susternya drku ganti jadi...
3,@Baebbyz @isyour__crush @kochengfess Aku baru ...,"Aku baru adopt kucing kak, eh tbtb dia diar...",Aku baru adopt kucing kak eh tbtb dia diare mi...
4,capek banget bisa bisanya meluk dokternya😭\r\n...,capek banget bisa bisanya meluk dokternya😭\r\n...,capek banget bisa bisanya meluk dokternya Craz...
...,...,...,...
1740,RT @tutudetoxic: @Akashi2105 @lazy_homer @iams...,"RT : Tenang, perang di ukraina gak ngince...",Tenang perang di ukraina gak ngincer warga sip...
1741,@SecuLitty @AngelenosLos @ddubokki @MitaCestal...,Mama jawab ke dokternya klo mama dah ster...,Mama jawab ke dokternya klo mama dah steril Ma...
1742,#Halodoc scam banget udh bayar pake debit aja ...,#Halodoc scam banget udh bayar pake debit aja ...,Halodoc scam banget udh bayar pake debit aja m...
1743,"kmrn janjian dokter langsung buat hari selasa,...","kmrn janjian dokter langsung buat hari selasa,...",kmrn janjian dokter langsung buat hari selasa ...


In [14]:
# casefolding
def caseFolding(s):
    newStrTweetCaseFold = s.lower()
    
    return newStrTweetCaseFold
data['tweet_case_folding'] = data['tweet_cleaning'].apply(caseFolding)
data

,fulltext,tweet_remove_user,tweet_cleaning,tweet_case_folding
0,@bucinbabang @bertanyarl sebelumnya aku konsul...,sebelumnya aku konsultasi dulu ke dokter yg ...,sebelumnya aku konsultasi dulu ke dokter yg du...,sebelumnya aku konsultasi dulu ke dokter yg du...
1,@bika_engine @adams69apple @Blackgownn @tangan...,Serius ms glow? Gua kira produk skincare y...,Serius ms glow Gua kira produk skincare yg kas...,serius ms glow gua kira produk skincare yg kas...
2,"udah lama ga ke limi, dateng2 susternya drku g...","udah lama ga ke limi, dateng2 susternya drku g...",udah lama ga ke limi susternya drku ganti jadi...,udah lama ga ke limi susternya drku ganti jadi...
3,@Baebbyz @isyour__crush @kochengfess Aku baru ...,"Aku baru adopt kucing kak, eh tbtb dia diar...",Aku baru adopt kucing kak eh tbtb dia diare mi...,aku baru adopt kucing kak eh tbtb dia diare mi...
4,capek banget bisa bisanya meluk dokternya😭\r\n...,capek banget bisa bisanya meluk dokternya😭\r\n...,capek banget bisa bisanya meluk dokternya Craz...,capek banget bisa bisanya meluk dokternya craz...
...,...,...,...,...
1740,RT @tutudetoxic: @Akashi2105 @lazy_homer @iams...,"RT : Tenang, perang di ukraina gak ngince...",Tenang perang di ukraina gak ngincer warga sip...,tenang perang di ukraina gak ngincer warga sip...
1741,@SecuLitty @AngelenosLos @ddubokki @MitaCestal...,Mama jawab ke dokternya klo mama dah ster...,Mama jawab ke dokternya klo mama dah steril Ma...,mama jawab ke dokternya klo mama dah steril ma...
1742,#Halodoc scam banget udh bayar pake debit aja ...,#Halodoc scam banget udh bayar pake debit aja ...,Halodoc scam banget udh bayar pake debit aja m...,halodoc scam banget udh bayar pake debit aja m...
1743,"kmrn janjian dokter langsung buat hari selasa,...","kmrn janjian dokter langsung buat hari selasa,...",kmrn janjian dokter langsung buat hari selasa ...,kmrn janjian dokter langsung buat hari selasa ...


In [16]:
# tahap tokenizing
def tweetTokenization(s):
    tokens = word_tokenize(s)

    return tokens
data['tweet_tokenization'] = data['tweet_case_folding'].apply(tweetTokenization)
data

,fulltext,tweet_remove_user,tweet_cleaning,tweet_case_folding,tweet_tokenization
0,@bucinbabang @bertanyarl sebelumnya aku konsul...,sebelumnya aku konsultasi dulu ke dokter yg ...,sebelumnya aku konsultasi dulu ke dokter yg du...,sebelumnya aku konsultasi dulu ke dokter yg du...,"[sebelumnya, aku, konsultasi, dulu, ke, dokter..."
1,@bika_engine @adams69apple @Blackgownn @tangan...,Serius ms glow? Gua kira produk skincare y...,Serius ms glow Gua kira produk skincare yg kas...,serius ms glow gua kira produk skincare yg kas...,"[serius, ms, glow, gua, kira, produk, skincare..."
2,"udah lama ga ke limi, dateng2 susternya drku g...","udah lama ga ke limi, dateng2 susternya drku g...",udah lama ga ke limi susternya drku ganti jadi...,udah lama ga ke limi susternya drku ganti jadi...,"[udah, lama, ga, ke, limi, susternya, drku, ga..."
3,@Baebbyz @isyour__crush @kochengfess Aku baru ...,"Aku baru adopt kucing kak, eh tbtb dia diar...",Aku baru adopt kucing kak eh tbtb dia diare mi...,aku baru adopt kucing kak eh tbtb dia diare mi...,"[aku, baru, adopt, kucing, kak, eh, tbtb, dia,..."
4,capek banget bisa bisanya meluk dokternya😭\r\n...,capek banget bisa bisanya meluk dokternya😭\r\n...,capek banget bisa bisanya meluk dokternya Craz...,capek banget bisa bisanya meluk dokternya craz...,"[capek, banget, bisa, bisanya, meluk, dokterny..."
...,...,...,...,...,...
1740,RT @tutudetoxic: @Akashi2105 @lazy_homer @iams...,"RT : Tenang, perang di ukraina gak ngince...",Tenang perang di ukraina gak ngincer warga sip...,tenang perang di ukraina gak ngincer warga sip...,"[tenang, perang, di, ukraina, gak, ngincer, wa..."
1741,@SecuLitty @AngelenosLos @ddubokki @MitaCestal...,Mama jawab ke dokternya klo mama dah ster...,Mama jawab ke dokternya klo mama dah steril Ma...,mama jawab ke dokternya klo mama dah steril ma...,"[mama, jawab, ke, dokternya, klo, mama, dah, s..."
1742,#Halodoc scam banget udh bayar pake debit aja ...,#Halodoc scam banget udh bayar pake debit aja ...,Halodoc scam banget udh bayar pake debit aja m...,halodoc scam banget udh bayar pake debit aja m...,"[halodoc, scam, banget, udh, bayar, pake, debi..."
1743,"kmrn janjian dokter langsung buat hari selasa,...","kmrn janjian dokter langsung buat hari selasa,...",kmrn janjian dokter langsung buat hari selasa ...,kmrn janjian dokter langsung buat hari selasa ...,"[kmrn, janjian, dokter, langsung, buat, hari, ..."


In [17]:
# stopwords
list_stopwords = stopwords.words('indonesian')

list_stopwords.extend(["dg", "ny","d", "klo",
                      "amp", "krn", "n", 'u', 'jd', "nyg",
                       "hehe", "nder", "der", "pen", "sis", "jg",
                       "bgt", 'dah', 'ni', 'so', 'x', 'ri', 'dos', 'eee',
                       'skrng', 'skr', 'kpd', 'j', 's', 'b', 'jgn2', 'gara2',
                       'utk', 'y', 'g', 'm', 'pm', 't', 'dm', 'rm', 'p', 'indonesi', 'https',
                       'ampe', 'rt'
                      ])

list_stopwords = set(list_stopwords)

def removeStopwords(words):
    return ' '.join([word for word in words if word not in list_stopwords])
data['tweet_stopwords'] = data['tweet_tokenization'].apply(removeStopwords)
data

,fulltext,tweet_remove_user,tweet_cleaning,tweet_case_folding,tweet_tokenization,tweet_stopwords
0,@bucinbabang @bertanyarl sebelumnya aku konsul...,sebelumnya aku konsultasi dulu ke dokter yg ...,sebelumnya aku konsultasi dulu ke dokter yg du...,sebelumnya aku konsultasi dulu ke dokter yg du...,"[sebelumnya, aku, konsultasi, dulu, ke, dokter...",konsultasi dokter yg nanganin kena gbs doktern...
1,@bika_engine @adams69apple @Blackgownn @tangan...,Serius ms glow? Gua kira produk skincare y...,Serius ms glow Gua kira produk skincare yg kas...,serius ms glow gua kira produk skincare yg kas...,"[serius, ms, glow, gua, kira, produk, skincare...",serius ms glow gua produk skincare yg dokterny...
2,"udah lama ga ke limi, dateng2 susternya drku g...","udah lama ga ke limi, dateng2 susternya drku g...",udah lama ga ke limi susternya drku ganti jadi...,udah lama ga ke limi susternya drku ganti jadi...,"[udah, lama, ga, ke, limi, susternya, drku, ga...",udah ga limi susternya drku ganti blahbloh
3,@Baebbyz @isyour__crush @kochengfess Aku baru ...,"Aku baru adopt kucing kak, eh tbtb dia diar...",Aku baru adopt kucing kak eh tbtb dia diare mi...,aku baru adopt kucing kak eh tbtb dia diare mi...,"[aku, baru, adopt, kucing, kak, eh, tbtb, dia,...",adopt kucing kak eh tbtb diare minggi kejadia ...
4,capek banget bisa bisanya meluk dokternya😭\r\n...,capek banget bisa bisanya meluk dokternya😭\r\n...,capek banget bisa bisanya meluk dokternya Craz...,capek banget bisa bisanya meluk dokternya craz...,"[capek, banget, bisa, bisanya, meluk, dokterny...",capek banget bisanya meluk dokternya crazylove
...,...,...,...,...,...,...
1740,RT @tutudetoxic: @Akashi2105 @lazy_homer @iams...,"RT : Tenang, perang di ukraina gak ngince...",Tenang perang di ukraina gak ngincer warga sip...,tenang perang di ukraina gak ngincer warga sip...,"[tenang, perang, di, ukraina, gak, ngincer, wa...",tenang perang ukraina gak ngincer warga sipil
1741,@SecuLitty @AngelenosLos @ddubokki @MitaCestal...,Mama jawab ke dokternya klo mama dah ster...,Mama jawab ke dokternya klo mama dah steril Ma...,mama jawab ke dokternya klo mama dah steril ma...,"[mama, jawab, ke, dokternya, klo, mama, dah, s...",mama dokternya mama steril mama nanya emng knp...
1742,#Halodoc scam banget udh bayar pake debit aja ...,#Halodoc scam banget udh bayar pake debit aja ...,Halodoc scam banget udh bayar pake debit aja m...,halodoc scam banget udh bayar pake debit aja m...,"[halodoc, scam, banget, udh, bayar, pake, debi...",halodoc scam banget udh bayar pake debit aja d...
1743,"kmrn janjian dokter langsung buat hari selasa,...","kmrn janjian dokter langsung buat hari selasa,...",kmrn janjian dokter langsung buat hari selasa ...,kmrn janjian dokter langsung buat hari selasa ...,"[kmrn, janjian, dokter, langsung, buat, hari, ...",kmrn janjian dokter langsung selasa trs dikaba...


In [18]:
#Ambil data yang sudah dipreprocessing

data['tweet_cleans'] = data['tweet_stopwords']
data

,fulltext,tweet_remove_user,tweet_cleaning,tweet_case_folding,tweet_tokenization,tweet_stopwords,tweet_cleans
0,@bucinbabang @bertanyarl sebelumnya aku konsul...,sebelumnya aku konsultasi dulu ke dokter yg ...,sebelumnya aku konsultasi dulu ke dokter yg du...,sebelumnya aku konsultasi dulu ke dokter yg du...,"[sebelumnya, aku, konsultasi, dulu, ke, dokter...",konsultasi dokter yg nanganin kena gbs doktern...,konsultasi dokter yg nanganin kena gbs doktern...
1,@bika_engine @adams69apple @Blackgownn @tangan...,Serius ms glow? Gua kira produk skincare y...,Serius ms glow Gua kira produk skincare yg kas...,serius ms glow gua kira produk skincare yg kas...,"[serius, ms, glow, gua, kira, produk, skincare...",serius ms glow gua produk skincare yg dokterny...,serius ms glow gua produk skincare yg dokterny...
2,"udah lama ga ke limi, dateng2 susternya drku g...","udah lama ga ke limi, dateng2 susternya drku g...",udah lama ga ke limi susternya drku ganti jadi...,udah lama ga ke limi susternya drku ganti jadi...,"[udah, lama, ga, ke, limi, susternya, drku, ga...",udah ga limi susternya drku ganti blahbloh,udah ga limi susternya drku ganti blahbloh
3,@Baebbyz @isyour__crush @kochengfess Aku baru ...,"Aku baru adopt kucing kak, eh tbtb dia diar...",Aku baru adopt kucing kak eh tbtb dia diare mi...,aku baru adopt kucing kak eh tbtb dia diare mi...,"[aku, baru, adopt, kucing, kak, eh, tbtb, dia,...",adopt kucing kak eh tbtb diare minggi kejadia ...,adopt kucing kak eh tbtb diare minggi kejadia ...
4,capek banget bisa bisanya meluk dokternya😭\r\n...,capek banget bisa bisanya meluk dokternya😭\r\n...,capek banget bisa bisanya meluk dokternya Craz...,capek banget bisa bisanya meluk dokternya craz...,"[capek, banget, bisa, bisanya, meluk, dokterny...",capek banget bisanya meluk dokternya crazylove,capek banget bisanya meluk dokternya crazylove
...,...,...,...,...,...,...,...
1740,RT @tutudetoxic: @Akashi2105 @lazy_homer @iams...,"RT : Tenang, perang di ukraina gak ngince...",Tenang perang di ukraina gak ngincer warga sip...,tenang perang di ukraina gak ngincer warga sip...,"[tenang, perang, di, ukraina, gak, ngincer, wa...",tenang perang ukraina gak ngincer warga sipil,tenang perang ukraina gak ngincer warga sipil
1741,@SecuLitty @AngelenosLos @ddubokki @MitaCestal...,Mama jawab ke dokternya klo mama dah ster...,Mama jawab ke dokternya klo mama dah steril Ma...,mama jawab ke dokternya klo mama dah steril ma...,"[mama, jawab, ke, dokternya, klo, mama, dah, s...",mama dokternya mama steril mama nanya emng knp...,mama dokternya mama steril mama nanya emng knp...
1742,#Halodoc scam banget udh bayar pake debit aja ...,#Halodoc scam banget udh bayar pake debit aja ...,Halodoc scam banget udh bayar pake debit aja m...,halodoc scam banget udh bayar pake debit aja m...,"[halodoc, scam, banget, udh, bayar, pake, debi...",halodoc scam banget udh bayar pake debit aja d...,halodoc scam banget udh bayar pake debit aja d...
1743,"kmrn janjian dokter langsung buat hari selasa,...","kmrn janjian dokter langsung buat hari selasa,...",kmrn janjian dokter langsung buat hari selasa ...,kmrn janjian dokter langsung buat hari selasa ...,"[kmrn, janjian, dokter, langsung, buat, hari, ...",kmrn janjian dokter langsung selasa trs dikaba...,kmrn janjian dokter langsung selasa trs dikaba...


In [19]:
#hapus column yang tidak dibutuhkan
data.drop(data.columns[[0,1,2,3,4,5]], axis = 1, inplace = True)
data

,tweet_cleans
0,konsultasi dokter yg nanganin kena gbs doktern...
1,serius ms glow gua produk skincare yg dokterny...
2,udah ga limi susternya drku ganti blahbloh
3,adopt kucing kak eh tbtb diare minggi kejadia ...
4,capek banget bisanya meluk dokternya crazylove
...,...
1740,tenang perang ukraina gak ngincer warga sipil
1741,mama dokternya mama steril mama nanya emng knp...
1742,halodoc scam banget udh bayar pake debit aja d...
1743,kmrn janjian dokter langsung selasa trs dikaba...


In [20]:
#menghapus duplikat kalimat pada dataset
data.drop_duplicates(subset = "tweet_cleans", keep = "first", inplace=True)
data

,tweet_cleans
0,konsultasi dokter yg nanganin kena gbs doktern...
1,serius ms glow gua produk skincare yg dokterny...
2,udah ga limi susternya drku ganti blahbloh
3,adopt kucing kak eh tbtb diare minggi kejadia ...
4,capek banget bisanya meluk dokternya crazylove
...,...
1739,pemalu tu butuh kerja yauda deh yg diperiksa d...
1741,mama dokternya mama steril mama nanya emng knp...
1742,halodoc scam banget udh bayar pake debit aja d...
1743,kmrn janjian dokter langsung selasa trs dikaba...


**TWEET CLEAN INTO TWEET ENGLISH**

In [21]:
from google_trans_new import google_translator  
translator = google_translator()  

def convert_eng(tweet):
  return tweet

  translator.translate(tweet,lang_tgt='en')

data['tweet_english'] = data['tweet_cleans'].apply(convert_eng)
data

,tweet_cleans,tweet_english
0,konsultasi dokter yg nanganin kena gbs doktern...,konsultasi dokter yg nanganin kena gbs doktern...
1,serius ms glow gua produk skincare yg dokterny...,serius ms glow gua produk skincare yg dokterny...
2,udah ga limi susternya drku ganti blahbloh,udah ga limi susternya drku ganti blahbloh
3,adopt kucing kak eh tbtb diare minggi kejadia ...,adopt kucing kak eh tbtb diare minggi kejadia ...
4,capek banget bisanya meluk dokternya crazylove,capek banget bisanya meluk dokternya crazylove
...,...,...
1739,pemalu tu butuh kerja yauda deh yg diperiksa d...,pemalu tu butuh kerja yauda deh yg diperiksa d...
1741,mama dokternya mama steril mama nanya emng knp...,mama dokternya mama steril mama nanya emng knp...
1742,halodoc scam banget udh bayar pake debit aja d...,halodoc scam banget udh bayar pake debit aja d...
1743,kmrn janjian dokter langsung selasa trs dikaba...,kmrn janjian dokter langsung selasa trs dikaba...


In [22]:
import googletrans
from googletrans import *
translator = googletrans.Translator()

data['tweet_english'] = data['tweet_english'].astype(str) #changing datatype to string
data['tweet_english'] = data['tweet_cleans'].apply(translator.translate, src='auto', dest='en').apply(getattr, args=('text',))
data

,tweet_cleans,tweet_english
0,konsultasi dokter yg nanganin kena gbs doktern...,"consulting the doctor who handled the GBS, the..."
1,serius ms glow gua produk skincare yg dokterny...,my ms glow series is a skincare product whose ...
2,udah ga limi susternya drku ganti blahbloh,"there's no limit for the sister, I'm changing ..."
3,adopt kucing kak eh tbtb diare minggi kejadia ...,"adopt a cat, sis, eh, tbtb, the diary of the w..."
4,capek banget bisanya meluk dokternya crazylove,"really tired, usually hugging the doctor crazy..."
...,...,...
1739,pemalu tu butuh kerja yauda deh yg diperiksa d...,"Shyness needs work, yes, what the doctor check..."
1741,mama dokternya mama steril mama nanya emng knp...,"mom, the doctor, mom is sterile, mom asks why ..."
1742,halodoc scam banget udh bayar pake debit aja d...,"hellodoc it's really a scam, I just paid with ..."
1743,kmrn janjian dokter langsung selasa trs dikaba...,I immediately made a doctor's appointment on T...


**LABEL WITH LIBRARY TEXTBLOB**

In [23]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from nltk.stem import PorterStemmer
from nltk.tokenize import sent_tokenize, word_tokenize

In [24]:
ps = PorterStemmer() 

def stemming_data(x):
    return ps.stem(x)

data['tweet_english'] = data['tweet_english'].apply(stemming_data)

In [25]:
data_tweet = list(data['tweet_english'])
polaritas = 0

status = []
total_positif = total_negatif = total_netral = total = 0

for i, tweet in enumerate(data_tweet):
    analysis = TextBlob(tweet)
    polaritas += analysis.polarity

    if analysis.sentiment.polarity > 0.0:
        total_positif += 1
        status.append('Positif')
    elif analysis.sentiment.polarity == 0.0:
        total_netral += 1
        status.append('Netral')
    else:
        total_negatif += 1
        status.append('Negatif')

    total += 1 
    
print(f'Hasil Analisis Data:\nPositif = {total_positif}\nNetral = {total_netral}\nNegatif = {total_negatif}')
print(f'\nTotal Data : {total}')

Hasil Analisis Data:
Positif = 562
Netral = 578
Negatif = 427

Total Data : 1567


In [26]:
status = pd.DataFrame({'klasifikasi': status})
data['klasifikasi'] = status
data.tail()

,tweet_cleans,tweet_english,klasifikasi
1739,pemalu tu butuh kerja yauda deh yg diperiksa d...,"shyness needs work, yes, what the doctor check...",NaN
1741,mama dokternya mama steril mama nanya emng knp...,"mom, the doctor, mom is sterile, mom asks why ...",NaN
1742,halodoc scam banget udh bayar pake debit aja d...,"hellodoc it's really a scam, i just paid with ...",NaN
1743,kmrn janjian dokter langsung selasa trs dikaba...,i immediately made a doctor's appointment on t...,NaN
1744,anak cepet sembuh patah retak tulang drpd dewa...,children recover from fractures and fractures ...,NaN


**SAVE THE DATASET AFTER PREPROCESSING**

In [27]:
data.to_excel('./dataset-tambahan-preprocessing.xlsx', encoding='utf8', index=True)